In [ ]:
import sys, os
# Imports of vbn_utils, ccf_utils, decoding_utils, analysis_utils,
# notebook_utils, vbn_4day_utils resolve via vbn_code/utilities/.
_UTILS_DIR = os.path.abspath(os.path.join(os.getcwd(), '..', 'utilities'))
if _UTILS_DIR not in sys.path:
    sys.path.insert(0, _UTILS_DIR)

import h5py
import pandas as pd
import numpy as np
import os
from pathlib import Path
from matplotlib import pyplot as plt
%matplotlib inline 
from functools import partial
from ccf_utils import get_area_color
import vbn_utils
import decoding_utils as du

In [ ]:
#Paths to all of the useful supplemental tables and tensors
active_tensor_file = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/vbnAllUnitSpikeTensor.hdf5"
passive_tensor_file = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/vbnAllUnitSpikeTensor_passive.hdf5"
opto_tensor_file = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/vbn_opto_tensor_unit_grouped.hdf5"

stim_table_file = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/master_stim_table_no_filter.csv"
unit_table_file = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/master_units_with_responsiveness.csv"
unit_table_with_rf_stats = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/units_with_rf_stats.csv"
unit_table_opto_metrics = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/unit_opto_metrics.csv"
unit_cluster_labels = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/unit_cluster_labels.csv"
unit_vis_responsive = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/units_with_vis_responsiveness.csv"

sessions_table_file = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/vbn_s3_cache/visual-behavior-neuropixels-0.4.0/project_metadata/ecephys_sessions.csv"

In [ ]:
structure_tree = pd.read_csv("/Volumes/programs/mindscope/workgroups/np-behavior/ccf_structure_tree_2017.csv")

### Generated on HPC by 'run_pooled_decoding.py'

In [ ]:
decoding_results_base = "/Volumes/programs/mindscope/workgroups/np-behavior/VBN_decoding_from_sensory_action_clusters"
label_to_dir = {'image': 'pooledImageDecoding',
                'lick': 'pooledLickDecoding',
                'change': 'pooledChangeDecoding',
                'changeprechange': 'pooledChangePrechangeDecoding',
                'reaction_time': 'pooledReactionTimeDecoding',
                'visual_response': 'pooledVisualResponseDecoding',}

In [ ]:
high_res = True
if high_res:
    plt.rcParams['figure.dpi'] = 150
    plt.rcParams['savefig.dpi'] = 300
    plt.rcParams['font.size'] = 12
    plt.rcParams['pdf.fonttype'] = 42
    
    plt.rcParams['figure.facecolor'] = 'white'
    plt.rcParams['axes.facecolor'] = 'white'
    plt.rcParams['savefig.facecolor'] = 'white'  # affects clipboard copy too
    plt.rcParams['savefig.transparent'] = False

In [ ]:
from notebook_utils import (
    bootstrap_CI,
    normalize,
    upsample,
    plot_decoding_results,
    plot_facemap_decoding,
    get_latency2,
    fitCurve,
    calc_gompertz,
    invert_gompertz,
    get_sigmoidfit_midpoint,
    get_latency_sigmoid_fit
)

## Session decoding for LP and VISp

In [ ]:
session_change_dir = "/Volumes/programs/mindscope/workgroups/np-behavior/VBN_decoding_from_sensory_action_clusters/sessionChangeDecoding_basesub"
unit_sample_size = 10

qualifying_files = [f for f in os.listdir(session_change_dir) if f.endswith(f'[{unit_sample_size}].npy')]
qualifying_sessions = [f.split('_')[1] for f in qualifying_files if 'LP' in f or 'VISp' in f]
sessions_with_LP_and_VISp = []
for session in qualifying_sessions:
    session_files = [f for f in qualifying_files if f.split('_')[1] == session and ('LP' in f or 'VISp' in f)]
    if len([s for s in session_files if 'LP' in s])>0 and len([s for s in session_files if 'VISp' in s])>0:
        sessions_with_LP_and_VISp.append(session_files)

sessions_with_LP_and_VISp = np.unique(sessions_with_LP_and_VISp)

In [ ]:
len(sessions_with_LP_and_VISp)

In [ ]:
summary_dict = {'LP':[], 'VISp':[]}
for session_file in sessions_with_LP_and_VISp:
    full_path = os.path.join(session_change_dir, session_file)
    data = np.load(full_path)
    
    area = 'LP' if 'LP' in session_file else 'VISp'
    print(f'{area} : {data.shape}')
    summary_dict[area].append(np.mean(data, axis=0))


In [ ]:
binsize = 10
time = np.arange(binsize, 750+binsize, binsize)

In [ ]:
lp_lats = get_latency2(np.stack(summary_dict['LP']), 0.5, time, slice(0,10), upsample_factor=10, norm=True, threshold=0.5, sub='chance')[1]
visp_lats = get_latency2(np.stack(summary_dict['VISp']), 0.5, time, slice(0,10), upsample_factor=10, norm=True, threshold=0.5, sub='chance')[1]

fig, ax = plt.subplots()
ax.plot(lp_lats, visp_lats, 'ko')
ax.plot([45,65], [45,65], 'k--')
ax.set_aspect('equal')
ax.set_xlabel('LP latency')
ax.set_ylabel('VISp latency')
vbn_utils.formatFigure(fig, ax)

nonans = ~np.isnan(lp_lats) & ~np.isnan(visp_lats)
p = scipy.stats.wilcoxon(lp_lats[nonans], visp_lats[nonans])[1]
ax.text(50, 45, 'p = {:.2e}'.format(p))

print(np.sum(nonans))


In [ ]:
plt.rcParams['font.size'] = 18
binsize = 10
time = np.arange(binsize, 750+binsize, binsize)
lp_array = np.stack(summary_dict['LP'])
visp_array = np.stack(summary_dict['VISp'])
above_thresh_lp = np.max(lp_array[:,:10], axis=1) > 0.6
above_thresh_visp = np.max(visp_array[:,:10], axis=1) > 0.6

above_thresh = above_thresh_lp & above_thresh_visp

fig, ax = plt.subplots()
ax.plot(time, (lp_array[above_thresh]).T, color='r', alpha=0.1)
ax.plot(time, (visp_array[above_thresh]).T, color='b', alpha=0.1)
ax.plot(time, (lp_array[above_thresh]).mean(axis=0), color='r', label='LP')
ax.plot(time, (visp_array[above_thresh]).mean(axis=0), color='b', label='VISp')
ax.legend()
ax.set_xlim(0,100)
ax.set_ylim(0.4, 0.8)


fig, ax = plt.subplots()
vbn_utils.mean_sem_plot(lp_array[above_thresh], ax, time, color=get_area_color('LP', structure_tree), label='LP')
vbn_utils.mean_sem_plot(visp_array[above_thresh], ax, time, color=get_area_color('VISp', structure_tree), label='VISp')
ax.set_xlim(0,120)
ax.set_ylim(0.4, 0.75)
ax.axhline(0.5, color='k', ls='dotted')
ax.set_xlabel('Time from stimulus (ms)')
ax.set_ylabel('Decoding Accuracy')
ax.legend(frameon=False)
vbn_utils.formatFigure(fig, ax)
print(np.sum(above_thresh), np.sum(above_thresh))

## Timing of image, change and lick decoding across areas

In [ ]:
decoding_results_base = "/Volumes/programs/mindscope/workgroups/np-behavior/VBN_decoding_from_sensory_action_clusters/with_unitsamp_replacement_and_same_splits_for_sessionunits"

basesub = '_basesub'
label_to_dir = {'image': f'pooledImageDecoding{basesub}',
                'lick': f'pooledLickDecoding{basesub}',
                'change': f'pooledChangeDecoding{basesub}',
                'changeprechange': f'pooledChangePrechangeDecoding{basesub}',
                'reaction_time': f'pooledReactionTimeDecoding{basesub}',
                'visual_response': f'pooledVisualResponseDecoding{basesub}',}


labels = ['change','image', 'lick', 'reaction_time', 'visual_response']
labels = ['change', 'image', 'lick']
clusters = ['sensory', 'action',]

areas = ['LGd', 'LP', 'VISall', 'SCMRN', 'Hipp', 'VISp', 'VISl', 'VISal', 'VISrl', 'VISam', 'VISpm',]

unitSampleSizes = [50,100,]
nPsuedoFlashes = [100,]
nUnitSamples = [1000,]
binsize = 10

def binsize_filter(filename, binssize):
    if binsize==5:
        return '5binsize' in filename
    else:
        return '5binsize' not in filename

decoding_dict_basesub = {cond: {l: {a: {c: {uss: {npf: {nus: [] for nus in nUnitSamples} for npf in nPsuedoFlashes} 
                                            for uss in unitSampleSizes} for c in clusters} 
                                            for a in areas} for l in labels} 
                                            for cond in ['active', 'passive']}
for cond in ['active', 'passive']:
    for label in labels:
        dir = label_to_dir[label] + '_' + cond
        decoding_results_dir = os.path.join(decoding_results_base, dir)
        decoding_results = os.listdir(decoding_results_dir)

        for area in areas:

            area_data_files = [f for f in decoding_results if (f.split('_')[1]==area) and binsize_filter(f, binsize)]
            for cluster in clusters:
                
                area_cluster_data = [f for f in area_data_files if f.split('_')[2]==cluster]
                
                for uss in unitSampleSizes:
                    area_cluster_uss = [f for f in area_cluster_data if f.split('_')[3]==str(uss)]

                    for npf in nPsuedoFlashes:
                        area_cluster_uss_npf = [f for f in area_cluster_uss if f.split('_')[4]==str(npf)]

                        for nus in nUnitSamples:
                            area_cluster_uss_npf_nus = [f for f in area_cluster_uss_npf if f.replace('.npy', '').split('_')[5]==str(nus) in f]
                            if len(area_cluster_uss_npf_nus) == 1:
                                decoding_dict_basesub[cond][label][area][cluster][uss][npf][nus] = np.load(os.path.join(decoding_results_dir, area_cluster_uss_npf_nus[0]))


### Load facemap data

In [ ]:
import h5py

f = h5py.File("/Volumes/programs/mindscope/workgroups/np-behavior/VBN_video_analysis/facemapData/1044385384_524761_20200819.behavior_proc.hdf5", 'r')

In [ ]:
data_dir = "/Volumes/programs/mindscope/workgroups/np-behavior/VBN_video_analysis/facemapDecoding_basesub"
session_files = os.listdir(data_dir)

all_session_data = []
for session_file in session_files:
    session_data = np.load(os.path.join(data_dir, session_file), allow_pickle=True)
    session_data = session_data.item()
    all_session_data.append(session_data['balancedAccuracy']['non-change lick'])


In [ ]:
all_session_data = [a for a in all_session_data if len(a)>0]

### calculate latencies

In [ ]:
# suppress warnings
import warnings
warnings.filterwarnings("ignore")

In [ ]:
from vbn_utils import formatFigure
plt.rcParams['font.size'] = 14
neuron_num = 100
bootstrap_num = 1000
areas = ['LGd', 'LP', 'VISall', 'VISp', 'VISl', 'VISal', 'VISrl', 'VISam', 'VISpm', 'SCMRN', 'Hipp', ]

latency_summary = {area: {cluster:{} for cluster in ['sensory', 'action']} for area in areas + ['facemap']}
time = np.arange(binsize, 750+binsize, binsize)
latency_threshold = 0.5
for label in ['image', 'change', 'lick',]:

    fig, ax = plt.subplots(1,2)
    fig.suptitle(label)
    fig.set_size_inches(8,3.8)

    cluster = 'sensory' if label != 'lick' else 'action'
    lims = [0,int(100/binsize)] if label != 'lick' else [0, int(750/binsize)]
    axlims = np.array(lims)*binsize
    chance = 1/8 if label=='image' else 0.5
    for i, area in enumerate(areas):
        if len(decoding_dict_basesub['active'][label][area][cluster][neuron_num][100][bootstrap_num]) == 0:
            continue

        tup, latencies, mean_latency, low_error, hi_error = get_latency2(decoding_dict_basesub['active'][label][area][cluster][neuron_num][100][bootstrap_num], 
                                                                   chance, time, slice(*lims), threshold=latency_threshold, upsample_factor=10, sub='min', norm=True)

        latencies = latencies.astype(float)
        
        if not area in ['VISp', 'VISl', 'VISal', 'VISrl', 'VISam', 'VISpm']:
            plot_decoding_results(decoding_dict_basesub, label, area, cluster, neuron_num, 100, bootstrap_num, time,
                                bootstrap_iterations=100, condition='active', ax=ax[0], norm=False, norm_slice=slice(*lims),
                                plotlabel=area, lw=2)
            
        
        print(f'{area} {cluster} {label} latency: {np.nanmean(latencies)}')
        
        latency_summary[area][cluster][label] = {'mean': mean_latency, 'low': low_error, 'high': hi_error, 'latencies': latencies}
        ax[1].errorbar(i, mean_latency, yerr=[[low_error], [hi_error]], color=get_area_color(area, structure_tree), fmt='o')

    if label == 'lick':
        frameInterval = 1/60
        facemap_time = np.arange(0,0.75+frameInterval/2,frameInterval)*1000
        tup, latencies, mean_latency, low_error, hi_error = get_latency2(np.array(all_session_data), 0.5, facemap_time, slice(0,46), threshold=latency_threshold, upsample_factor=17)
        latency_summary['facemap']['action']['lick'] = {'mean': mean_latency, 'low': low_error, 'high': hi_error}
        ax[1].errorbar(len(areas), mean_latency, yerr=[[low_error], [hi_error]], color='k', fmt='o')
        ax1_x_upper_lim = len(areas)+0.5
        ax1_y_lims = (50, 250)
        print(f'facemap {label} latency: {mean_latency}')
    else:
        ax1_x_upper_lim = len(areas)-0.5
        ax1_y_lims = (28, 80)


    if label == 'image':
        ax0_y_lims = (0.08, 1)
    else:
        ax0_y_lims = (0.45, 1)        
    
    

    ax[1].set_xlim(-0.5, ax1_x_upper_lim)
    ax[1].set_ylim(ax1_y_lims)
    ax[1].set_xticks(np.arange(len(areas)))
    ax[1].set_xticklabels(areas, rotation=45)
    ax[0].set_xlim(*axlims)
    ax[0].set_ylim(*ax0_y_lims)
    ax[0].axhline(chance, color='k', ls='--', alpha=0.5)

    ax[1].set_ylabel('Time to half max (ms)')
    ax[0].set_ylabel('Decoding Accuracy')

    xlabel = label if label != 'lick' else 'change'
    formatFigure(fig, ax[0], xLabel=f'Time from {xlabel} onset (ms)')
    formatFigure(fig, ax[1])
    plt.tight_layout()
    

### Generated on HPC by 'run_decoder_area_comparison_monte_carlo.py'

In [ ]:
## Load monte carlo null distribution for LP-VISp
null_data_dir = "/Volumes/programs/mindscope/workgroups/np-behavior/VBN_revision_decoder_area_comparison_nulls/pooledChangeDecoding_basesub_active"
null_files = os.listdir(null_data_dir)

num_permutations = 1000
null_values = []
for perm in range(num_permutations):
    v1_null_file = [f for f in null_files if f'pop1_{perm}_' in f]
    v1_null_data = np.load(os.path.join(null_data_dir, v1_null_file[0]))
    v1_null_latencies = get_latency2(v1_null_data, 0.5, time, slice(*[0,int(100/binsize)]), threshold=0.5, upsample_factor=10, sub='min', norm=True)[1]

    lp_null_file = [f for f in null_files if f'pop2_{perm}_' in f]
    lp_null_data = np.load(os.path.join(null_data_dir, lp_null_file[0]))
    lp_null_latencies = get_latency2(lp_null_data, 0.5, time, slice(*[0,int(100/binsize)]), threshold=0.5, upsample_factor=10, sub='min', norm=True)[1]

    null_values.append(np.nanmean(v1_null_latencies - lp_null_latencies))


In [ ]:
v1_lats = latency_summary['VISp']['sensory']['change']['latencies']
lp_lats = latency_summary['LP']['sensory']['change']['latencies']

obs = np.nanmean(v1_lats - lp_lats)

In [ ]:
pvalue = (1 + np.sum(np.abs(null_values) >= np.abs(obs))) / (num_permutations+1)
print(pvalue)

In [ ]:
plt.hist(null_values, align='mid')
plt.axvline(obs, color='k')

In [ ]:
## Load monte carlo null distribution for SCMRN/VISall action neurons (lick decoding)
null_data_dir = "/Volumes/programs/mindscope/workgroups/np-behavior/VBN_revision_decoder_area_comparison_nulls/pooledLickDecoding_basesub_active"
null_files = os.listdir(null_data_dir)

num_permutations = 1000
null_values = []
for perm in range(num_permutations):
    visall_null_file = [f for f in null_files if f'pop1_{perm}_' in f]
    visall_null_data = np.load(os.path.join(null_data_dir, visall_null_file[0]))
    visall_null_latencies = get_latency2(visall_null_data, 0.5, time, slice(*[0, int(750/binsize)]), threshold=0.5, upsample_factor=10, sub='min', norm=True)[1]

    scmrn_null_file = [f for f in null_files if f'pop2_{perm}_' in f]
    scmrn_null_data = np.load(os.path.join(null_data_dir, scmrn_null_file[0]))
    scmrn_null_latencies = get_latency2(scmrn_null_data, 0.5, time, slice(*[0, int(750/binsize)]), threshold=0.5, upsample_factor=10, sub='min', norm=True)[1]

    null_values.append(np.nanmean(visall_null_latencies - scmrn_null_latencies))


In [ ]:
visall_lats = latency_summary['VISall']['action']['lick']['latencies']
scmrn_lats = latency_summary['SCMRN']['action']['lick']['latencies']

obs = np.nanmean(visall_lats - scmrn_lats)

In [ ]:
pvalue = (1 + np.sum(np.abs(null_values) >= np.abs(obs))) / (num_permutations+1)
print(pvalue)

In [ ]:
plt.rcParams['font.size'] = 12
plt.hist(null_values, align='mid', color='gray', label='null distribution')
plt.axvline(obs, color='r', label='observed difference')
plt.xlabel('$\\Delta$ lick decoding latency (VIS - SC/MRN) (ms)')
plt.ylabel('Count')
plt.xlim(-65, 65)
plt.legend(frameon=False, loc='upper left')
plt.tight_layout()
vbn_utils.formatFigure(plt.gcf(), plt.gca())


In [ ]:
from analysis_utils import fitCurve, calc_gompertz

opto_results = (np.load("/Volumes/programs/mindscope/workgroups/np-behavior/VBN_opto/mouse_opto_fa_rates.npy"), 
                np.load("/Volumes/programs/mindscope/workgroups/np-behavior/VBN_opto/mouse_opto_hit_rates.npy"),
                np.load("/Volumes/programs/mindscope/workgroups/np-behavior/VBN_opto/opto_time.npy"))
opto_hit_rates = opto_results[1]
opto_time = opto_results[2]
opto_time[-1] = 150
fig, ax = plt.subplots()
for rates, color in zip([opto_results[0], opto_results[1]], ['r', 'b']):
    sigmoid_fits = [fitCurve(calc_gompertz, opto_time[:-1], rate[:-1], 
                           [rate.max(), 0.1, 75, rate.min()],
                           bounds=([rate.max()*0.9, -np.inf, -np.inf, -np.inf], [rate.max()*1.1, np.inf, np.inf, np.inf]))
                           for rate in list(rates) + [rates.mean(axis=0)]]
    sigmoids = [calc_gompertz(np.arange(-10, 140), *sigmoid_fit) for sigmoid_fit in sigmoid_fits]
    ax.plot(opto_time, rates.mean(axis=0), color + 'o', ms=8)
    ax.plot(np.arange(-10, 140), sigmoids[-1], color)

            
    mean = np.mean(rates, axis=0)
    sem = np.std(rates, axis=0)/(len(rates)**0.5)
    ax.errorbar(opto_time, mean, yerr = sem, color=color, linestyle='None')
    ax.plot(opto_time[-1], mean[-1], 'o', color=color, mfc='w', ms=8)

ax.set_xlabel('Laser onset relative to change (ms)')
ax.set_ylabel('Hit rate')
ax.set_xticks(np.arange(0, 200, 50))
ax.set_xticklabels(list(np.arange(0, 200, 50)[:-1]) + ['no opto'])

opto_latencies = []
for rate, sigmoid in zip(opto_hit_rates, sigmoids):
    opto_midpoint = sigmoid.min() + (sigmoid.max() - sigmoid.min())*0.2
    opto_latencies.append(np.argmin(np.abs(sigmoid - opto_midpoint)))

ax.axvline(np.median(opto_latencies), color='b', linestyle='--')


In [ ]:
fig, ax_dummy = plt.subplots()

In [ ]:
stim_table = pd.read_csv(stim_table_file)
units = pd.read_csv(unit_table_file)

In [ ]:
session_list = [str(s) for s in stim_table['ecephys_session_id'].unique()]

In [ ]:
unit_filter = du.getUnitsInRegion(units, 'VISall', rs=True) & du.apply_unit_quality_filter(units) & du.get_units_in_cluster(units, *np.arange(6))
unit_ids = units.loc[unit_filter]['unit_id'].values
session_list = [str(s) for s in units.loc[unit_filter]['ecephys_session_id'].unique()]
psth_data = vbn_utils.unit_averaged_psth(active_tensor_file, stim_table, session_list, unit_ids, *['engaged', 'is_change', 'lickbout_for_flash_during_response_window'])
passive_psth_data = vbn_utils.unit_averaged_psth(passive_tensor_file, stim_table, session_list, unit_ids, *['engaged', 'is_change', 'lickbout_for_flash_during_response_window'])

In [ ]:
all_psth_mean = np.concatenate(psth_data[0]).mean(axis=0)
passive_all_psth_mean = np.concatenate(passive_psth_data[0]).mean(axis=0)

all_psth_mean = all_psth_mean - all_psth_mean[:100].min()
passive_all_psth_mean = passive_all_psth_mean - passive_all_psth_mean[:100].min()

max_val = np.max([all_psth_mean.max(), passive_all_psth_mean.max()])

all_psth_norm = all_psth_mean/max_val
passive_all_psth_norm = passive_all_psth_mean/max_val

fig, ax = plt.subplots()
vbn_utils.mean_sem_plot(np.concatenate(psth_data[0]), color='b', ax=ax)
vbn_utils.mean_sem_plot(np.concatenate(passive_psth_data[0]), color='gray', ax=ax)

plt.xlim(50, 250)

In [ ]:
## aggregate timing plots

fig, ax = plt.subplots()
opto_sigmoid = calc_gompertz(np.arange(-10, 250), *sigmoid_fits[-1])
image_mean = plot_decoding_results(decoding_dict_basesub, 'image', 'LGd', 'sensory', 100, 100, 100, time,
                              bootstrap_iterations=100, ax=ax_dummy, norm=False, norm_slice=slice(*[0,int(150/binsize)]), return_mean=True)
change_mean = plot_decoding_results(decoding_dict_basesub, 'change', 'VISall', 'sensory', 100, 100, 100, time,
                              bootstrap_iterations=100, ax=ax_dummy, norm=False, norm_slice=slice(*[0,int(150/binsize)]), return_mean=True)
scmrn_lick_mean = plot_decoding_results(decoding_dict_basesub, 'lick', 'SCMRN', 'action', 100, 100, 100, time,
                              bootstrap_iterations=100, ax=ax_dummy, norm=False, norm_slice=slice(*[0,int(750/binsize)]), return_mean=True)

facemap_mean = plot_facemap_decoding(all_session_data, facemap_time, ax_dummy, bootstrap_iterations=100, color='k', return_mean=True)


normed = [(a[:lim] - a[:lim].min())/(max(a[:lim]-a[:lim].min())) for a, lim in zip([opto_sigmoid, change_mean, 
                                                                                    scmrn_lick_mean, image_mean, facemap_mean], [None, 50, 50, 50, 30])]
ax.plot(np.arange(-10, 250), normed[0], 'b', label='opto hit rates')
ax.plot(time[:25], normed[1][:25], 'r', label='VISall change decoding')
ax.plot(time[:25], normed[2][:25], color=get_area_color('MB', structure_tree), label='SCMRN lick decoding')
ax.plot(time[:25], normed[3][:25], color=get_area_color('LGd', structure_tree), label='LGd image decoding')
ax.plot(facemap_time[:16], normed[4][:16], 'k', label='facemap decoding')
#ax.plot(all_psth_norm[50:], 'gray')
# ax.fill_between(np.arange(250), all_psth_norm[50:300], np.zeros(250), color=get_area_color('VIS', structure_tree), alpha=0.7, label='VISall PSTH')

ax.set_xlim(0, 250)
# ax.legend()
formatFigure(fig, ax, xLabel='Time from change onset (ms)', yLabel='Normalized')
ax.plot([20, 100], [1.05, 1.05], 'gray', lw=5)

fig.savefig(os.path.join("/Volumes/programs/mindscope/workgroups/np-behavior/VBN Manuscript/Fig3 timing_figure", 'timing_aggregate_test.pdf'))

In [ ]:
from notebook_utils import set_spine_linewidth

In [ ]:
areas = ['LGd', 'LP', 'VISall', 'VISp', 'VISl', 'VISal', 'VISrl', 'VISam', 'VISpm', 'SCMRN', ]
area_labels = ['LGd', 'LP', 'VISall', 'VISp', 'VISl', 'VISal', 'VISrl', 'VISam', 'VISpm', 'SCm/MRN', ]

fig, ax = plt.subplots()
fig.set_size_inches(12,3)
fighist, axhist = plt.subplots(2,1)
for ia, area in enumerate(areas):
    for il, label in enumerate(['image', 'change']):
        if not label in latency_summary[area]['sensory']:
            continue
        mean = latency_summary[area]['sensory'][label]['mean']
        hi_error = latency_summary[area]['sensory'][label]['high']
        low_error = latency_summary[area]['sensory'][label]['low']
        if label=='change':
            fill_color = 'w'
            edge_color = get_area_color(area, structure_tree)
            size = 12
        else:
            fill_color = get_area_color(area, structure_tree)
            edge_color = 'none'
            size = 12
        ax.errorbar(mean, ia, xerr=[[low_error], [hi_error]], color=get_area_color(area, structure_tree), fmt='o', mfc=fill_color, ms=size, mec=edge_color, alpha=0.75, markeredgewidth=2)
        axhist[il].hist(latency_summary[area]['sensory'][label]['latencies'], histtype='step')

#ax.set_xlim(0,100)
ax.set_xlabel('Time from image onset to half max (ms)')
ax.set_yticks(np.arange(len(areas)))
ax.set_yticklabels(area_labels)

ax.set_xlim(30,80)
ax.set_xlabel('Time from image onset to half max (ms)')
ax.set_yticks(np.arange(len(areas)))
ax.set_yticklabels(area_labels)
ax.set_ylim(-0.6, len(areas))
vbn_utils.formatFigure(fig, ax)
set_spine_linewidth(ax, linewidth=2)
for ia, _ in enumerate(areas):
    if not ia%2==0:
        ax.axhspan(ia-0.5, ia+0.5, facecolor='lightgray', alpha=0.3, lw=0)

fig, ax = plt.subplots()
fig.set_size_inches(4,3)
fighist, axhist = plt.subplots()
areas = ['VISall', 'SCMRN']
for ys, area in zip([2,9], areas):# + ['facemap']):
    if not 'lick' in latency_summary[area]['action']:
            continue
    #multiplier = 10 if area != 'facemap' else 1
    mean = latency_summary[area]['action']['lick']['mean']
    hi_error = latency_summary[area]['action']['lick']['high']
    low_error = latency_summary[area]['action']['lick']['low']
    color = 'k' if area == 'facemap' else get_area_color(area, structure_tree)
    ax.errorbar(mean, ys, xerr=[[low_error], [hi_error]], color=color, fmt='>', ms=12, mec='none', markeredgewidth=2)
    axhist.hist(latency_summary[area]['action']['lick']['latencies'], histtype='step')

ax.set_xlim(100,250)
# ax.set_xlabel('Time from image onset to half max (ms)')
ax.set_ylim(0, 10)
ax.set_ylim(-0.6, 10)
ax.set_yticks([2,9])
ax.set_yticklabels(['VISall', 'SCm/MRN'])
vbn_utils.formatFigure(fig, ax)
ax.yaxis.set_visible(False)
ax.spines['left'].set_visible(False)  # Hides the left spine

set_spine_linewidth(ax, linewidth=2)

for ia in range(10):
    if not ia%2==0:
        ax.axhspan(ia-0.5, ia+0.5, facecolor='lightgray', alpha=0.3, lw=0)


In [ ]:
def get_rt2(row):
    
    trial_id = row['behavior_trial_id']
    session_id = row['ecephys_session_id']
    first_lick_in_trial = stim_table[(stim_table['behavior_trial_id']==trial_id)&(stim_table['ecephys_session_id']==session_id)]['lick_time'].values
    if len(first_lick_in_trial)==0:
        rt = np.nan 
    else:
        rt = np.nanmin(first_lick_in_trial) - row['start_time']

    return rt

lick_time_from_flash2 = stim_table[stim_table['is_change']].apply(get_rt2, axis=1)

In [ ]:
reaction_times = lick_time_from_flash2
fig, ax = plt.subplots()
ax.hist(reaction_times*1000, bins=np.arange(0, 1.01, 0.01)*1000, color='k', linewidth=2, alpha=0.3, ec='none', density=True)

## comparing decoding across action/sensory clusters

In [ ]:
from vbn_utils import formatFigure
plt.rcParams['font.size'] = 16
neuron_num = 100
num_bootstraps = 1000
areas = ['VISall', 'SCMRN',]

time = np.arange(binsize, 750+binsize, binsize)
latency_threshold = 0.5
for label in ['image', 'change', 'lick',]:

    fig, ax = plt.subplots(1,2)
    fig.suptitle(label)
    fig.set_size_inches(12,5)
    for cluster, ls in zip(['sensory', 'action'], ['-', '--']):


        lims = [0,int(300/binsize)] if label != 'lick' else [0, int(750/binsize)]
        axlims = np.array(lims)*binsize
        chance = 1/8 if label=='image' else 0.5
        for i, area in enumerate(areas):
            if len(decoding_dict_basesub['active'][label][area][cluster][neuron_num][100][num_bootstraps]) == 0:
                continue

            tup, latencies, mean_latency, low_error, hi_error = get_latency2(decoding_dict_basesub['active'][label][area][cluster][neuron_num][100][num_bootstraps], 
                                                                    chance, time, slice(*lims), threshold=latency_threshold, upsample_factor=10)
            
            
            plot_decoding_results(decoding_dict_basesub, label, area, cluster, neuron_num, 100, num_bootstraps, time,
                                bootstrap_iterations=100, condition='active', ax=ax[0], norm=False, norm_slice=slice(*lims),
                                plotlabel=area, ls=ls)
            
            
            print(f'{area} {cluster} {label} latency: {latencies.mean()}')
            
            ax[1].errorbar(i, mean_latency, yerr=[[low_error], [hi_error]], color=get_area_color(area, structure_tree), fmt='o')

        if label == 'lick':
            frameInterval = 1/60
            facemap_time = np.arange(0,0.75+frameInterval/2,frameInterval)*1000
            ax[1].errorbar(len(areas), mean_latency, yerr=[[low_error], [hi_error]], color='k', fmt='o')
            ax1_x_upper_lim = len(areas)+0.5
            ax1_y_lims = (50, 250)
            print(f'facemap {label} latency: {mean_latency}')
        else:
            ax1_x_upper_lim = len(areas)-0.5
            ax1_y_lims = (28, 80)


        if label == 'image':
            ax0_y_lims = (0.08, 1)
        else:
            ax0_y_lims = (0.45, 1)        
        
        
        ax[1].set_xlim(-0.5, ax1_x_upper_lim)
        ax[1].set_ylim(ax1_y_lims)
        ax[1].set_xticks(np.arange(len(areas)))
        ax[1].set_xticklabels(areas, rotation=45)
        ax[0].set_xlim(*axlims)
        ax[0].set_ylim(*ax0_y_lims)

        ax[0].set_ylabel('Decoding Accuracy')

        xlabel = label if label != 'lick' else 'change'
        formatFigure(fig, ax[0], xLabel=f'Time from {xlabel} onset (ms)')
        formatFigure(fig, ax[1])

## Decoding dropouts and sufficiency tests

### Generated on HPC by 'run_decoding_dropouts.py'

In [ ]:
change_decoding_dir = "/Volumes/programs/mindscope/workgroups/np-behavior/VBN_revision_decoding_dropouts/pooledChangeDecoding_basesub_active"
image_decoding_dir = "/Volumes/programs/mindscope/workgroups/np-behavior/VBN_revision_decoding_dropouts/pooledImageDecoding_basesub_active"

In [ ]:
decoding_results_base = "/Volumes/programs/mindscope/workgroups/np-behavior/VBN_revision_decoding_dropouts"
label_to_dir = {'image': 'pooledImageDecoding_basesub_active',
                'lick': 'pooledLickDecoding_basesub_active',
                'change': 'pooledChangeDecoding_basesub_active',
                'changeprechange': 'pooledChangePrechangeDecoding_basesub_active',
                'reaction_time': 'pooledReactionTimeDecoding_basesub_active',
                'visual_response': 'pooledVisualResponseDecoding_basesub_active',}

In [ ]:
decoding_dir = os.path.join(decoding_results_base, label_to_dir['change'])
subset_files = [f for f in decoding_files if 'subset' in f and 'dropout' in f]
subsets = [f.split('subset_')[1].split(f'_{test}')[0] for f in subset_files]
subsets

In [ ]:
time = np.arange(binsize, 750+binsize, binsize)

decoding_dir = os.path.join(decoding_results_base, label_to_dir['change'])
decoding_files = os.listdir(decoding_dir)
subset_files = [f for f in decoding_files if 'subset' in f and 'dropout' in f]
subsets = [f.split('subset_')[1].split(f'_dropout')[0] for f in subset_files]

decoding_labels = ['change', 'image']

for unitsubset in subsets:
    for decoding_label in decoding_labels:
        fig, axes = plt.subplots(1,2)
        fig.set_size_inches(12,4)

        for ax, test in zip(axes, ['dropout', 'sufficiency']):

            this_decoding_dir = os.path.join(decoding_results_base, label_to_dir[decoding_label])
            these_decoding_files = os.listdir(this_decoding_dir)
            subset_file = [f for f in these_decoding_files if f'subset_{unitsubset}' in f and test in f]

            if len(subset_file)==0:
                continue

            unitset = subset_file[0].split('set_')[1].split('_sub')[0]
            subset_data = np.load(os.path.join(this_decoding_dir, subset_file[0]))

            fig.suptitle(f"{decoding_label} \n Subset: {unitsubset}, Full: {unitset}")

            full_file = [f for f in these_decoding_files if unitset in f and '_full_' in f]

            if len(full_file)==0:
                continue

            full_data = np.load(os.path.join(this_decoding_dir, full_file[0]))

            
            ax.plot(time, subset_data.mean(axis=0), label= test.capitalize())
            ax.plot(time, full_data.mean(axis=0), label='Full')
            ax.legend()
            ax.set_title(f'{test.capitalize()}')
            ax.set_xlim(0, 300)
        
        plt.tight_layout()

In [ ]:
fam = np.load("/Volumes/programs/mindscope/workgroups/np-behavior/VBN_revision_decoding_dropouts/pooledImageDecoding_basesub_active/pooledImageDecoding_set_VISall_all_all_sensory_Familiar_full_100_100_100_10binsize.npy")
nov = np.load("/Volumes/programs/mindscope/workgroups/np-behavior/VBN_revision_decoding_dropouts/pooledImageDecoding_basesub_active/pooledImageDecoding_set_VISall_all_all_sensory_Novel_full_100_100_100_10binsize.npy")

In [ ]:
plt.plot(fam.mean(axis=0))
plt.plot(nov.mean(axis=0))
plt.xlim(0,15)